In [21]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import RXGate, RYGate, CZGate, MCXGate, RZGate, MCMT

import numpy as np

In [12]:
N = 2
B = 2
R = 2

In [16]:
rx_circuit = QuantumCircuit(N + R + 1)

rx_circuit.h(list(range(N + 1, N + 1 + R)))

# apply Rx rotation gates
for i in range(N):
	for r in range(R):
		gate = RXGate(np.pi / (2**r))
		rx_circuit.append(gate.control(1), (N + 1 + r, i))

# apply CNOT gate to target
rx_circuit.append(MCXGate(N), list(range(0, N+1)))

rx_circuit.draw()

┌───────┐┌─────────┐                         
q_0: ─────┤ Rx(π) ├┤ Rx(π/2) ├──────────────────────■──
          └───┬───┘└────┬────┘┌───────┐┌─────────┐  │  
q_1: ─────────┼─────────┼─────┤ Rx(π) ├┤ Rx(π/2) ├──■──
              │         │     └───┬───┘└────┬────┘┌─┴─┐
q_2: ─────────┼─────────┼─────────┼─────────┼─────┤ X ├
     ┌───┐    │         │         │         │     └───┘
q_3: ┤ H ├────■─────────┼─────────■─────────┼──────────
     ├───┤              │                   │          
q_4: ┤ H ├──────────────■───────────────────■──────────
     └───┘

In [19]:
transition_circ = QuantumCircuit(2*N + 1)
transition_circ.h(2*N)

for i in range(N):
	transition_circ.append(MCXGate(2), (i, 2*N, N + i))

transition_circ.draw()

q_0: ───────■───────
            │       
q_1: ───────┼────■──
          ┌─┴─┐  │  
q_2: ─────┤ X ├──┼──
          └─┬─┘┌─┴─┐
q_3: ───────┼──┤ X ├
     ┌───┐  │  └─┬─┘
q_4: ┤ H ├──■────■──
     └───┘

In [24]:
# R0lstar circuit
transformation_circ = QuantumCircuit(N + R + 1)

# append X gates
transformation_circ.x(list(range(N+1, N+R+1)))

# apply CZ gate
cz = RZGate(np.pi)
indices = list(range(0, N)) + list(range(N+1, N+R+1)) + [N]
transformation_circ.append(MCMT(cz, N+R, 1), indices)

# undo X gates
transformation_circ.x(list(range(N+1, N+R+1)))

transformation_circ.draw()

/var/folders/f0/g9bnvrbs7t15psm7b2d81gnm0000gn/T/ipykernel_27496/4227786876.py:10: DeprecationWarning: The class ``qiskit.circuit.library.generalized_gates.mcmt.MCMT`` is deprecated as of qiskit 1.4. It will be removed no earlier than 3 months after the release date. Use MCMTGate instead.
  transformation_circ.append(MCMT(cz, N+R, 1), indices)


┌───────┐     
q_0: ─────┤0      ├─────
          │       │     
q_1: ─────┤1      ├─────
          │       │     
q_2: ─────┤4 mcmt ├─────
     ┌───┐│       │┌───┐
q_3: ┤ X ├┤2      ├┤ X ├
     ├───┤│       │├───┤
q_4: ┤ X ├┤3      ├┤ X ├
     └───┘└───────┘└───┘

In [30]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
 
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

c = QuantumCircuit(5)

indices = list(range(0, N)) + list(range(N+1, N+R+1)) + [N]
c.append(MCMT(cz, N+R, 1), indices)

backend = FakeBrisbane()

pm = generate_preset_pass_manager(backend=backend)
phys = pm.run(c)

/var/folders/f0/g9bnvrbs7t15psm7b2d81gnm0000gn/T/ipykernel_27496/592064754.py:10: DeprecationWarning: The class ``qiskit.circuit.library.generalized_gates.mcmt.MCMT`` is deprecated as of qiskit 1.4. It will be removed no earlier than 3 months after the release date. Use MCMTGate instead.
  c.append(MCMT(cz, N+R, 1), indices)


In [35]:
phys.draw(idle_wires=False)

global phase: 5π/4
             ┌───┐                                                             »
q_4 -> 17 ───┤ X ├─────────────────────────────────────────────────────────────»
          ┌──┴───┴───┐┌────┐┌─────────┐                                        »
q_3 -> 29 ┤ Rz(-π/2) ├┤ √X ├┤ Rz(π/2) ├────────────────────────────────────────»
          └──┬───┬───┘└────┘└─────────┘┌──────┐                                »
q_1 -> 30 ───┤ X ├─────────────────────┤0     ├────────────────────────────────»
          ┌──┴───┴───┐┌────┐┌─────────┐│  Ecr │┌────┐┌──────────┐┌────┐┌──────┐»
q_2 -> 31 ┤ Rz(-π/2) ├┤ √X ├┤ Rz(π/2) ├┤1     ├┤ √X ├┤ Rz(-π/4) ├┤ √X ├┤0     ├»
          ├──────────┤├────┤├─────────┤└──────┘└────┘└──────────┘└────┘│  Ecr │»
q_0 -> 32 ┤ Rz(-π/2) ├┤ √X ├┤ Rz(π/2) ├────────────────────────────────┤1     ├»
          └──────────┘└────┘└─────────┘                                └──────┘»
«                                                                            »
«q_4 -> 17 ──────────────────────────────────────────────────────────────────»
«                                                                            »
«q_3 -> 29 ──────────────────────────────────────────────────────────────────»
«                                                                    ┌──────┐»
«q_1 -> 30 ──────────────────────────────────────────────────────────┤0     ├»
«          ┌──────────────┐┌────┐┌─────────────┐┌────┐┌─────────────┐│  Ecr │»
«q_2 -> 31 ┤ Rz(-0.52547) ├┤ √X ├┤ Rz(-1.1059) ├┤ √X ├┤ Rz(0.65822) ├┤1     ├»
«          └─┬──────────┬─┘├────┤└┬────────────┤├────┤└─┬──────────┬┘└──────┘»
«q_0 -> 32 ──┤ Rz(-π/2) ├──┤ √X ├─┤ Rz(-2.006) ├┤ √X ├──┤ Rz(-π/2) ├─────────»
«            └──────────┘  └────┘ └────────────┘└────┘  └──────────┘         »
«                                                                          »
«q_4 -> 17 ────────────────────────────────────────────────────────────────»
«                                                                          »
«q_3 -> 29 ────────────────────────────────────────────────────────────────»
«               ┌───┐     ┌──────────┐                                     »
«q_1 -> 30 ─────┤ X ├─────┤ Rz(-π/4) ├─────────────────────────────────────»
«          ┌────┴───┴────┐└──┬────┬──┘┌──────────────┐┌────┐┌─────────────┐»
«q_2 -> 31 ┤ Rz(0.56511) ├───┤ √X ├───┤ Rz(-0.99229) ├┤ √X ├┤ Rz(-2.2824) ├»
«          └─────────────┘   └────┘   └──────────────┘└────┘└─────────────┘»
«q_0 -> 32 ────────────────────────────────────────────────────────────────»
«                                                                          »
«                                                                          »
«q_4 -> 17 ────────────────────────────────────────────────────────────────»
«                                                                          »
«q_3 -> 29 ────────────────────────────────────────────────────────────────»
«                                                                          »
«q_1 -> 30 ────────────────────────────────────────────────────────────────»
«          ┌──────┐┌────────────┐┌────┐  ┌──────────┐ ┌────┐┌─────────────┐»
«q_2 -> 31 ┤0     ├┤ Rz(2.5261) ├┤ √X ├──┤ Rz(-π/3) ├─┤ √X ├┤ Rz(-2.5261) ├»
«          │  Ecr │└┬─────────┬─┘├────┤┌─┴──────────┴┐└────┘└─────────────┘»
«q_0 -> 32 ┤1     ├─┤ Rz(π/2) ├──┤ √X ├┤ Rz(-1.8871) ├─────────────────────»
«          └──────┘ └─────────┘  └────┘└─────────────┘                     »
«                                                                        »
«q_4 -> 17 ──────────────────────────────────────────────────────────────»
«                                                                        »
«q_3 -> 29 ──────────────────────────────────────────────────────────────»
«          ┌──────┐┌──────────┐┌────┐┌────────┐┌──────┐┌─────────┐ ┌────┐»
«q_1 -> 30 ┤0     ├┤ Rz(-π/2) ├┤ √X ├┤ Rz(-π) ├┤0     ├┤ Rz(π/2) ├─┤ √X ├»
«          │  Ecr │├─────────┬┘├────┤└────────┘│  Ecr │├─────────┴┐└────┘»
«q